# 90_tabelas_graficos_artigo

Este notebook gera tabelas e graficos de apoio ao artigo a partir da base bruta do PRONAF.

Objetivo desta entrega:
- construir a serie anual do PRONAF entre 2015 e 2025;
- somar o valor anual da coluna monetaria do programa;
- calcular a quantidade de CPFs unicos atendidos em cada ano;
- produzir um grafico de linhas com eixo duplo;
- salvar a tabela agregada e a figura em disco para reutilizacao no artigo.

Decisao metodologica importante:
- para o valor anual do credito, usa-se apenas o filtro `CD_PROGRAMA == "0001"`;
- nao se aplica filtro adicional de CPF para somar `VL_PARC_CREDITO`;
- o filtro de CPF valido e utilizado apenas para contar CPFs unicos atendidos por ano.

## 1. Imports e configuracao

Lemos a base bruta do PRONAF com colunas minimas, identificamos automaticamente a coluna monetaria existente no parquet e definimos os caminhos de saida da tabela e da figura.

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "dados" / "dados_brutos" / "pronaf.parquet"
OUTPUT_TABLE_PATH = ROOT / "dados" / "dados_processados" / "pronaf_evolucao_2015_2025.parquet"
OUTPUT_FIG_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "pronaf_evolucao_2015_2025.png"

DATA_REFERENCIA = pd.Timestamp("2025-12-31")
ANO_INICIAL = 2015
ANO_FINAL = 2025

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {INPUT_PATH}")

schema_cols = set(pq.ParquetFile(INPUT_PATH).schema.names)
colunas_valor_candidatas = ["VL_PARC_CREDITO", "VL_OPERACAO"]
VALOR_COL = next((c for c in colunas_valor_candidatas if c in schema_cols), None)
if VALOR_COL is None:
    raise ValueError(
        "Nenhuma coluna de valor encontrada entre as candidatas: "
        f"{colunas_valor_candidatas}"
    )

COLUNAS_PRONAF = ["CD_CPF_CNPJ", "ANO", VALOR_COL, "CD_PROGRAMA"]

print(f"ROOT: {ROOT}")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"OUTPUT_TABLE_PATH: {OUTPUT_TABLE_PATH}")
print(f"OUTPUT_FIG_PATH: {OUTPUT_FIG_PATH}")
print(f"Janela temporal: {ANO_INICIAL}-{ANO_FINAL} | referencia: {DATA_REFERENCIA.date()}")
print(f"Coluna monetaria utilizada: {VALOR_COL}")

## 2. Leitura da base bruta

Lemos somente as colunas necessarias para reduzir uso de memoria. Nesta etapa registramos `shape`, amostra inicial e `df.info()` para auditoria.

In [ ]:
df_pronaf = pd.read_parquet(
    INPUT_PATH,
    columns=COLUNAS_PRONAF,
    engine="pyarrow",
)

print(f"Shape inicial: {df_pronaf.shape}")
display(df_pronaf.head(3))
df_pronaf.info(show_counts=True)

## 3. Filtro do programa e preparo da janela temporal

A base para valor anual do credito usa apenas o filtro do programa `0001`, que identifica o PRONAF. Em seguida, convertemos `ANO` e a coluna monetaria para formato numerico e restringimos a visualizacao ao intervalo 2015-2025.

In [ ]:
linhas_iniciais = len(df_pronaf)

df_pronaf["CD_PROGRAMA"] = (
    df_pronaf["CD_PROGRAMA"]
    .astype("string")
    .str.strip()
    .str.zfill(4)
)

df_pronaf_programa = df_pronaf[df_pronaf["CD_PROGRAMA"] == "0001"].copy()
linhas_pos_programa = len(df_pronaf_programa)

df_pronaf_programa["ANO"] = pd.to_numeric(df_pronaf_programa["ANO"], errors="coerce")
anos_invalidos = int(df_pronaf_programa["ANO"].isna().sum())

mask_janela = df_pronaf_programa["ANO"].between(ANO_INICIAL, ANO_FINAL, inclusive="both")
df_pronaf_programa = df_pronaf_programa[mask_janela.fillna(False)].copy()

df_pronaf_programa[VALOR_COL] = pd.to_numeric(df_pronaf_programa[VALOR_COL], errors="coerce")
valores_invalidos = int(df_pronaf_programa[VALOR_COL].isna().sum())
df_pronaf_programa[VALOR_COL] = df_pronaf_programa[VALOR_COL].fillna(0.0)

print(f"Linhas iniciais: {linhas_iniciais:,}")
print(f"Apos filtro CD_PROGRAMA == '0001': {linhas_pos_programa:,}")
print(f"ANOs invalidos (coerce): {anos_invalidos:,}")
print(f"Shape da base de valor apos filtro temporal {ANO_INICIAL}-{ANO_FINAL}: {df_pronaf_programa.shape}")
print(f"{VALOR_COL} invalido/ausente tratado com 0.0: {valores_invalidos:,}")
display(df_pronaf_programa[["ANO", VALOR_COL]].describe().T)

## 4. Base para contagem de CPFs unicos

A contagem de cobertura anual por pessoa exige CPF consistente. Por isso, a partir da base do programa ja filtrada, padronizamos o identificador e mantemos apenas CPFs com 11 digitos para a metrica de pessoas unicas atendidas.

In [ ]:
df_pronaf_cpfs = df_pronaf_programa[["CD_CPF_CNPJ", "ANO"]].copy()

df_pronaf_cpfs["CD_CPF_CNPJ"] = (
    df_pronaf_cpfs["CD_CPF_CNPJ"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .str.strip()
    .str.zfill(11)
)

is_cpf = df_pronaf_cpfs["CD_CPF_CNPJ"].str.len().eq(11).fillna(False)
df_pronaf_cpfs = df_pronaf_cpfs[is_cpf].copy()
df_pronaf_cpfs = df_pronaf_cpfs.rename(columns={"CD_CPF_CNPJ": "cpf_agricultor"})
df_pronaf_cpfs["cpf_agricultor"] = df_pronaf_cpfs["cpf_agricultor"].astype("string")

print(f"Shape da base de CPFs validos: {df_pronaf_cpfs.shape}")
print(f"CPFs unicos na janela: {df_pronaf_cpfs['cpf_agricultor'].nunique():,}")
display(df_pronaf_cpfs.head(3))

## 5. Agregacao anual

A tabela anual combina duas metricas distintas: o valor total do PRONAF por ano, calculado com filtro apenas de programa, e a quantidade de CPFs unicos atendidos por ano, calculada na base de CPFs validos.

In [ ]:
df_valor_ano = (
    df_pronaf_programa.groupby("ANO", as_index=False)
    .agg(
        valor_total_credito=(VALOR_COL, "sum"),
        n_operacoes_programa=(VALOR_COL, "size"),
    )
)

df_cpfs_ano = (
    df_pronaf_cpfs.groupby("ANO", as_index=False)
    .agg(cpfs_unicos=("cpf_agricultor", "nunique"))
)

anos_referencia = pd.DataFrame({"ANO": list(range(ANO_INICIAL, ANO_FINAL + 1))})
df_evolucao = (
    anos_referencia
    .merge(df_valor_ano, on="ANO", how="left")
    .merge(df_cpfs_ano, on="ANO", how="left")
    .sort_values("ANO")
    .reset_index(drop=True)
)

df_evolucao["valor_total_credito"] = df_evolucao["valor_total_credito"].fillna(0.0)
df_evolucao["n_operacoes_programa"] = df_evolucao["n_operacoes_programa"].fillna(0).astype("int64")
df_evolucao["cpfs_unicos"] = df_evolucao["cpfs_unicos"].fillna(0).astype("int64")

print(f"Shape da tabela anual: {df_evolucao.shape}")
print("Missing nas colunas-chave:")
display(df_evolucao[["ANO", "valor_total_credito", "cpfs_unicos"]].isna().sum())
display(df_evolucao)

## 6. Grafico para o artigo

O eixo principal mostra o valor anual total do PRONAF, calculado com filtro apenas do programa. O eixo secundario mostra a quantidade de CPFs unicos atendidos no ano. Essa separacao deixa explicito que as duas series usam regras de construcao diferentes.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ---------------------------------------------------------
# Formatadores
# ---------------------------------------------------------
def format_bilhoes(x, _):
    return f"{x / 1_000_000_000:.1f} bi".replace(".", ",")

def format_milhoes(x, _):
    return f"{x / 1_000_000:.1f} mi".replace(".", ",")

# ---------------------------------------------------------
# Configuração geral
# ---------------------------------------------------------
plt.style.use("default")
fig, ax1 = plt.subplots(figsize=(10, 5.5))

# ---------------------------------------------------------
# Série 1: valor contratado
# ---------------------------------------------------------
linha_valor, = ax1.plot(
    df_evolucao["ANO"],
    df_evolucao["valor_total_credito"],
    color="#1b5e20",
    marker="o",
    linewidth=2.4,
    zorder=3,
    label="Valor contratado",
)

ax1.set_xlabel("Ano", fontsize=12)
ax1.set_ylabel("Valor contratado (bilhões)", color="#1b5e20", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#1b5e20", labelsize=11)
ax1.tick_params(axis="x", labelsize=11)
ax1.yaxis.set_major_formatter(FuncFormatter(format_bilhoes))
ax1.set_xticks(df_evolucao["ANO"])

# ---------------------------------------------------------
# Série 2: CPFs únicos atendidos
# ---------------------------------------------------------
ax2 = ax1.twinx()
linha_cpfs, = ax2.plot(
    df_evolucao["ANO"],
    df_evolucao["cpfs_unicos"],
    color="#0d47a1",
    marker="o",
    linestyle="--",
    linewidth=2.4,
    zorder=3,
    label="CPFs únicos atendidos",
)

ax2.set_ylabel("CPFs únicos atendidos (milhões)", color="#0d47a1", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#0d47a1", labelsize=11)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1_000_000:.1f}".replace(".", ",")))

# ---------------------------------------------------------
# Grid minimalista
# ---------------------------------------------------------
ax1.grid(axis="y", color="#e6e6e6", linewidth=0.7)
ax1.grid(axis="x", visible=False)

# ---------------------------------------------------------
# Remover molduras desnecessárias
# ---------------------------------------------------------
for spine in ["top", "right", "left"]:
    ax1.spines[spine].set_visible(False)

for spine in ["top", "left", "right"]:
    ax2.spines[spine].set_visible(False)

# ---------------------------------------------------------
# Legenda central
# ---------------------------------------------------------
fig.legend(
    handles=[linha_valor, linha_cpfs],
    labels=["Valor contratado", "CPFs únicos atendidos"],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.90),
    ncol=2,
    frameon=False,
    fontsize=11,
)

# ---------------------------------------------------------
# Destaques em 2021 e 2025
# ---------------------------------------------------------
anos_destaque = [2021, 2025]

for ano in anos_destaque:
    linha = df_evolucao.loc[df_evolucao["ANO"] == ano].iloc[0]

    x = linha["ANO"]
    y_valor = linha["valor_total_credito"]
    y_cpfs = linha["cpfs_unicos"]

    ax1.scatter(x, y_valor, s=70, color="#1b5e20", zorder=4)
    ax2.scatter(x, y_cpfs, s=70, color="#0d47a1", zorder=4)

    # CPF acima do ponto
    ax2.annotate(
        format_milhoes(y_cpfs, None),
        xy=(x, y_cpfs),
        xytext=(0, 12),
        textcoords="offset points",
        color="#0d47a1",
        fontsize=10,
        fontweight="semibold",
        ha="center",
        va="bottom",
    )

    # Valor abaixo do ponto
    ax1.annotate(
        format_bilhoes(y_valor, None),
        xy=(x, y_valor),
        xytext=(0, -12),
        textcoords="offset points",
        color="#1b5e20",
        fontsize=10,
        fontweight="semibold",
        ha="center",
        va="top",
    )

# ---------------------------------------------------------
# Ajuste final de layout
# ---------------------------------------------------------
fig.tight_layout()
fig.subplots_adjust(top=0.84)

if matplotlib.get_backend().lower() != "agg":
    plt.show()



## 7. Persistencia e checagens finais

Salvamos a tabela anual em parquet e exportamos a figura em PNG. Ao final, registramos `shape`, missing em colunas-chave e `df.info()` da tabela resumida.

In [ ]:
OUTPUT_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_PATH.parent.mkdir(parents=True, exist_ok=True)

df_evolucao.to_parquet(OUTPUT_TABLE_PATH, index=False)
fig.savefig(OUTPUT_FIG_PATH, dpi=300, bbox_inches="tight")

print(f"Tabela salva em: {OUTPUT_TABLE_PATH}")
print(f"Figura salva em: {OUTPUT_FIG_PATH}")
print(f"Shape final da tabela anual: {df_evolucao.shape}")
print("Missing em colunas-chave:")
display(df_evolucao[["ANO", "valor_total_credito", "cpfs_unicos"]].isna().sum())

print("Resumo final de tipos (df.info):")
df_evolucao.info(show_counts=True)

## 8. Tabela comparativa completa de AUCs das trilhas

Nesta secao consolidamos, em formato longo, **todos os valores de `AUC_s`** persistidos pelas trilhas A, B e C em `dados/dados_processados/outputs_pu`.

Objetivo editorial:
- reunir em uma unica tabela os AUCs de validacao, teste, sensibilidade de prior e calibracao;
- preservar a rastreabilidade do valor individual de AUC por componente e por artefato de origem;
- evitar mistura metodologica indevida por meio da coluna `familia_variacao`.

Ressalva metodologica obrigatoria:
- a metrica comparada aqui e `AUC_s`, isto e, separacao do rotulo observado `s`;
- essa AUC e uma **proxy operacional** e nao mede diretamente o desempenho em `y` latente;
- por isso, `modelo_principal`, `sensibilidade_prior` e `calibracao` permanecem explicitamente marcados na tabela final.


In [ ]:
OUTPUTS_PU_DIR = ROOT / "dados" / "dados_processados" / "outputs_pu"
OUTPUT_AUC_TABLE_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_auc_todas_as_trilhas.parquet"
OUTPUT_AUC_CSV_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_auc_todas_as_trilhas.csv"

def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f"Nenhum artefato encontrado entre: {paths}")

def safe_float(value):
    if pd.isna(value):
        return float("nan")
    return float(value)

def append_auc_row(rows, *, notebook_origem, trilha, componente, modelo, familia_variacao, estagio_auc, score_variant, auc_s, fonte_artefato, prior=float("nan"), observacao=""):
    rows.append(
        {
            "notebook_origem": notebook_origem,
            "trilha": trilha,
            "componente": componente,
            "modelo": modelo,
            "familia_variacao": familia_variacao,
            "estagio_auc": estagio_auc,
            "score_variant": score_variant,
            "prior": prior,
            "auc_s": safe_float(auc_s),
            "fonte_artefato": fonte_artefato,
            "observacao": observacao,
        }
    )

rows_auc = []

# ---------------------------------------------------------
# Trilha A: A2, A3 e A4 score final
# ---------------------------------------------------------
path_a = first_existing([
    OUTPUTS_PU_DIR / "trilhaA_metricas_resumo.parquet",
    OUTPUTS_PU_DIR / "trilhaA_metricas_resumo_smoke.parquet",
])
df_a = pd.read_parquet(path_a, engine="pyarrow")

for _, row in df_a.iterrows():
    componente = str(row["componente"])
    familia = "calibracao" if componente.startswith("A4") else "modelo_principal"
    score_variant = "score_final_operacional" if componente.startswith("A4") else "principal"
    observacao = "Trilha A: proxy supervisionado em s."

    append_auc_row(
        rows_auc,
        notebook_origem="20_trilha_A_proxy_supervisionado_s.ipynb",
        trilha="Trilha A",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_auc="validacao",
        score_variant=score_variant,
        auc_s=row.get("auc_s_val", float("nan")),
        fonte_artefato=path_a.name,
        observacao=observacao,
    )
    append_auc_row(
        rows_auc,
        notebook_origem="20_trilha_A_proxy_supervisionado_s.ipynb",
        trilha="Trilha A",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_auc="teste",
        score_variant=score_variant,
        auc_s=row.get("auc_s_test", float("nan")),
        fonte_artefato=path_a.name,
        observacao=observacao,
    )

# ---------------------------------------------------------
# Trilha B: B1, B2 e B3
# ---------------------------------------------------------
path_b = first_existing([
    OUTPUTS_PU_DIR / "trilhaB_comparacao_interna.parquet",
    OUTPUTS_PU_DIR / "trilhaB_comparacao_interna_smoke.parquet",
])
df_b = pd.read_parquet(path_b, engine="pyarrow")

for _, row in df_b.iterrows():
    componente = str(row["codigo_modelo"])
    familia = "calibracao" if componente == "B3" else "modelo_principal"
    observacao = "Trilha B: PU classico sob SCAR."

    append_auc_row(
        rows_auc,
        notebook_origem="21_trilha_B_pu_classico.ipynb",
        trilha="Trilha B",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_auc="validacao",
        score_variant=str(row.get("score_principal_componente", "principal")),
        auc_s=row.get("auc_proxy_s_val", float("nan")),
        fonte_artefato=path_b.name,
        observacao=observacao,
    )
    append_auc_row(
        rows_auc,
        notebook_origem="21_trilha_B_pu_classico.ipynb",
        trilha="Trilha B",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_auc="teste",
        score_variant=str(row.get("score_principal_componente", "principal")),
        auc_s=row.get("auc_proxy_s", float("nan")),
        fonte_artefato=path_b.name,
        observacao=observacao,
    )

# ---------------------------------------------------------
# Trilha C: C1 principal, C2 sensibilidade de prior e C3 calibracao
# ---------------------------------------------------------
path_c_metrics = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_metrics_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_metrics_summary_smoke.parquet",
])
path_c_prior = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_prior_sensitivity_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_prior_sensitivity_summary_smoke.parquet",
])
path_c_calibration = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_calibration_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_calibration_summary_smoke.parquet",
])

df_c_metrics = pd.read_parquet(path_c_metrics, engine="pyarrow")
df_c_prior = pd.read_parquet(path_c_prior, engine="pyarrow")
df_c_calibration = pd.read_parquet(path_c_calibration, engine="pyarrow")

for _, row in df_c_metrics.iterrows():
    append_auc_row(
        rows_auc,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C1",
        modelo=str(row["modelo"]),
        familia_variacao="modelo_principal",
        estagio_auc="teste",
        score_variant="principal",
        prior=safe_float(row.get("prior_principal", float("nan"))),
        auc_s=row.get("AUC_s", float("nan")),
        fonte_artefato=path_c_metrics.name,
        observacao="Trilha C: execucao principal do nnPU.",
    )

for _, row in df_c_prior.iterrows():
    prior_value = safe_float(row["prior"])
    append_auc_row(
        rows_auc,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C2",
        modelo="nnPU MLP (PyTorch)",
        familia_variacao="sensibilidade_prior",
        estagio_auc="teste",
        score_variant="principal",
        prior=prior_value,
        auc_s=row.get("AUC_s", float("nan")),
        fonte_artefato=path_c_prior.name,
        observacao=f"Trilha C: sensibilidade ao prior com prior={prior_value:.2f}.",
    )

stage_map = {"val": "validacao", "test": "teste", "fit": "fit", "apply": "apply"}
for _, row in df_c_calibration.iterrows():
    stage_raw = str(row.get("stage", "teste")).strip().lower()
    append_auc_row(
        rows_auc,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C3",
        modelo="nnPU MLP (PyTorch)",
        familia_variacao="calibracao",
        estagio_auc=stage_map.get(stage_raw, stage_raw),
        score_variant=str(row.get("score_variant", "principal")),
        prior=float("nan"),
        auc_s=row.get("AUC_s", float("nan")),
        fonte_artefato=path_c_calibration.name,
        observacao="Trilha C: comparacao entre score bruto e calibrado.",
    )

df_auc_todas_as_trilhas = pd.DataFrame(rows_auc)

trilha_order = {"Trilha A": 1, "Trilha B": 2, "Trilha C": 3}
familia_order = {"modelo_principal": 1, "sensibilidade_prior": 2, "calibracao": 3}
estagio_order = {"validacao": 1, "teste": 2, "fit": 3, "apply": 4}

df_auc_todas_as_trilhas["_trilha_ord"] = df_auc_todas_as_trilhas["trilha"].map(trilha_order).fillna(99)
df_auc_todas_as_trilhas["_familia_ord"] = df_auc_todas_as_trilhas["familia_variacao"].map(familia_order).fillna(99)
df_auc_todas_as_trilhas["_estagio_ord"] = df_auc_todas_as_trilhas["estagio_auc"].map(estagio_order).fillna(99)

df_auc_todas_as_trilhas = df_auc_todas_as_trilhas.sort_values(
    ["_trilha_ord", "_familia_ord", "componente", "prior", "_estagio_ord", "score_variant"],
    kind="stable",
).drop(columns=["_trilha_ord", "_familia_ord", "_estagio_ord"]).reset_index(drop=True)

display(df_auc_todas_as_trilhas)


## 9. Persistencia e checagens da tabela de AUCs

Persistimos a tabela consolidada em `outputs_artigo` e registramos checagens minimas de completude, missing e distribuicao por trilha e por familia metodologica.


In [ ]:
OUTPUT_AUC_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)

df_auc_todas_as_trilhas.to_parquet(OUTPUT_AUC_TABLE_PATH, index=False)
df_auc_todas_as_trilhas.to_csv(OUTPUT_AUC_CSV_PATH, index=False, encoding="utf-8")

print(f"Tabela de AUCs salva em: {OUTPUT_AUC_TABLE_PATH}")
print(f"Tabela de AUCs em CSV salva em: {OUTPUT_AUC_CSV_PATH}")
print(f"Shape final da tabela de AUCs: {df_auc_todas_as_trilhas.shape}")
print("Missing em colunas-chave:")
display(df_auc_todas_as_trilhas[["trilha", "componente", "familia_variacao", "estagio_auc", "auc_s"]].isna().sum())

print("Contagem por trilha:")
display(df_auc_todas_as_trilhas.groupby("trilha", dropna=False).size().rename("n_linhas").reset_index())

print("Contagem por familia_variacao:")
display(df_auc_todas_as_trilhas.groupby("familia_variacao", dropna=False).size().rename("n_linhas").reset_index())

print("Resumo final de tipos (df.info):")
df_auc_todas_as_trilhas.info(show_counts=True)


## 10. Tabela comparativa operacional de Recall@k

Nesta secao consolidamos, em formato longo, **todos os valores de recall** persistidos pelas trilhas A, B e C em `dados/dados_processados/outputs_pu`.

Objetivo editorial:
- comparar o desempenho operacional do ranking nos recortes mais priorizados;
- preservar a rastreabilidade do valor individual de recall por componente e por artefato de origem;
- manter a distincao entre `modelo_principal`, `sensibilidade_prior` e `calibracao`.

Ressalva metodologica obrigatoria:
- `Recall@k` e uma metrica operacional de priorizacao, isto e, mede quantos positivos observados em `s` aparecem entre os casos mais bem ranqueados;
- ela complementa a `AUC_s`, mas nao substitui sua leitura global;
- assim como a `AUC_s`, nao mede diretamente o desfecho latente `y`.


In [ ]:
OUTPUT_RECALL_TABLE_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_recall_todas_as_trilhas.parquet"
OUTPUT_RECALL_CSV_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_recall_todas_as_trilhas.csv"

def append_recall_row(rows, *, notebook_origem, trilha, componente, modelo, familia_variacao, estagio_metrica, score_variant, metrica_recall, valor_recall, fonte_artefato, prior=float("nan"), observacao=""):
    rows.append(
        {
            "notebook_origem": notebook_origem,
            "trilha": trilha,
            "componente": componente,
            "modelo": modelo,
            "familia_variacao": familia_variacao,
            "estagio_metrica": estagio_metrica,
            "score_variant": score_variant,
            "prior": prior,
            "metrica_recall": metrica_recall,
            "valor_recall": safe_float(valor_recall),
            "fonte_artefato": fonte_artefato,
            "observacao": observacao,
        }
    )

def append_recall_triplet(rows, *, notebook_origem, trilha, componente, modelo, familia_variacao, estagio_metrica, score_variant, fonte_artefato, recall_1pct, recall_5pct, recall_10pct, prior=float("nan"), observacao=""):
    metric_pairs = [
        ("Recall@1%", recall_1pct),
        ("Recall@5%", recall_5pct),
        ("Recall@10%", recall_10pct),
    ]
    for metrica_recall, valor_recall in metric_pairs:
        append_recall_row(
            rows,
            notebook_origem=notebook_origem,
            trilha=trilha,
            componente=componente,
            modelo=modelo,
            familia_variacao=familia_variacao,
            estagio_metrica=estagio_metrica,
            score_variant=score_variant,
            metrica_recall=metrica_recall,
            valor_recall=valor_recall,
            fonte_artefato=fonte_artefato,
            prior=prior,
            observacao=observacao,
        )

rows_recall = []

# ---------------------------------------------------------
# Trilha A: A2, A3 e A4 score final
# ---------------------------------------------------------
path_a_recall = first_existing([
    OUTPUTS_PU_DIR / "trilhaA_metricas_resumo.parquet",
    OUTPUTS_PU_DIR / "trilhaA_metricas_resumo_smoke.parquet",
])
df_a_recall = pd.read_parquet(path_a_recall, engine="pyarrow")

for _, row in df_a_recall.iterrows():
    componente = str(row["componente"])
    familia = "calibracao" if componente.startswith("A4") else "modelo_principal"
    score_variant = "score_final_operacional" if componente.startswith("A4") else "principal"
    append_recall_triplet(
        rows_recall,
        notebook_origem="20_trilha_A_proxy_supervisionado_s.ipynb",
        trilha="Trilha A",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_metrica="teste",
        score_variant=score_variant,
        prior=float("nan"),
        recall_1pct=row.get("recall_at_1pct_test", float("nan")),
        recall_5pct=row.get("recall_at_5pct_test", float("nan")),
        recall_10pct=row.get("recall_at_10pct_test", float("nan")),
        fonte_artefato=path_a_recall.name,
        observacao="Trilha A: recall de teste do proxy supervisionado em s.",
    )

# ---------------------------------------------------------
# Trilha B: B1, B2 e B3
# ---------------------------------------------------------
path_b_recall = first_existing([
    OUTPUTS_PU_DIR / "trilhaB_comparacao_interna.parquet",
    OUTPUTS_PU_DIR / "trilhaB_comparacao_interna_smoke.parquet",
])
df_b_recall = pd.read_parquet(path_b_recall, engine="pyarrow")

for _, row in df_b_recall.iterrows():
    componente = str(row["codigo_modelo"])
    familia = "calibracao" if componente == "B3" else "modelo_principal"
    append_recall_triplet(
        rows_recall,
        notebook_origem="21_trilha_B_pu_classico.ipynb",
        trilha="Trilha B",
        componente=componente,
        modelo=str(row["model_name"]),
        familia_variacao=familia,
        estagio_metrica="teste",
        score_variant=str(row.get("score_principal_componente", "principal")),
        prior=float("nan"),
        recall_1pct=row.get("recall_at_1pct", float("nan")),
        recall_5pct=row.get("recall_at_5pct", float("nan")),
        recall_10pct=row.get("recall_at_10pct", float("nan")),
        fonte_artefato=path_b_recall.name,
        observacao="Trilha B: recall de teste do PU classico sob SCAR.",
    )

# ---------------------------------------------------------
# Trilha C: C1 principal, C2 sensibilidade de prior e C3 calibracao
# ---------------------------------------------------------
path_c_metrics_recall = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_metrics_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_metrics_summary_smoke.parquet",
])
path_c_prior_recall = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_prior_sensitivity_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_prior_sensitivity_summary_smoke.parquet",
])
path_c_calibration_recall = first_existing([
    OUTPUTS_PU_DIR / "trilhaC_calibration_summary.parquet",
    OUTPUTS_PU_DIR / "trilhaC_calibration_summary_smoke.parquet",
])

df_c_metrics_recall = pd.read_parquet(path_c_metrics_recall, engine="pyarrow")
df_c_prior_recall = pd.read_parquet(path_c_prior_recall, engine="pyarrow")
df_c_calibration_recall = pd.read_parquet(path_c_calibration_recall, engine="pyarrow")

for _, row in df_c_metrics_recall.iterrows():
    append_recall_triplet(
        rows_recall,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C1",
        modelo=str(row["modelo"]),
        familia_variacao="modelo_principal",
        estagio_metrica="teste",
        score_variant="principal",
        prior=safe_float(row.get("prior_principal", float("nan"))),
        recall_1pct=row.get("Recall@1%", float("nan")),
        recall_5pct=row.get("Recall@5%", float("nan")),
        recall_10pct=row.get("Recall@10%", float("nan")),
        fonte_artefato=path_c_metrics_recall.name,
        observacao="Trilha C: execucao principal do nnPU.",
    )

for _, row in df_c_prior_recall.iterrows():
    prior_value = safe_float(row["prior"])
    append_recall_triplet(
        rows_recall,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C2",
        modelo="nnPU MLP (PyTorch)",
        familia_variacao="sensibilidade_prior",
        estagio_metrica="teste",
        score_variant="principal",
        prior=prior_value,
        recall_1pct=row.get("Recall@1%", float("nan")),
        recall_5pct=row.get("Recall@5%", float("nan")),
        recall_10pct=row.get("Recall@10%", float("nan")),
        fonte_artefato=path_c_prior_recall.name,
        observacao=f"Trilha C: sensibilidade ao prior com prior={prior_value:.2f}.",
    )

stage_map_recall = {"val": "validacao", "test": "teste", "fit": "fit", "apply": "apply"}
for _, row in df_c_calibration_recall.iterrows():
    stage_raw = str(row.get("stage", "teste")).strip().lower()
    append_recall_triplet(
        rows_recall,
        notebook_origem="22_trilha_C_nnpu.ipynb",
        trilha="Trilha C",
        componente="C3",
        modelo="nnPU MLP (PyTorch)",
        familia_variacao="calibracao",
        estagio_metrica=stage_map_recall.get(stage_raw, stage_raw),
        score_variant=str(row.get("score_variant", "principal")),
        prior=float("nan"),
        recall_1pct=row.get("Recall@1%", float("nan")),
        recall_5pct=row.get("Recall@5%", float("nan")),
        recall_10pct=row.get("Recall@10%", float("nan")),
        fonte_artefato=path_c_calibration_recall.name,
        observacao="Trilha C: comparacao entre score bruto e calibrado.",
    )

df_recall_todas_as_trilhas = pd.DataFrame(rows_recall)

trilha_order_recall = {"Trilha A": 1, "Trilha B": 2, "Trilha C": 3}
familia_order_recall = {"modelo_principal": 1, "sensibilidade_prior": 2, "calibracao": 3}
estagio_order_recall = {"validacao": 1, "teste": 2, "fit": 3, "apply": 4}
metrica_order_recall = {"Recall@1%": 1, "Recall@5%": 2, "Recall@10%": 3}

df_recall_todas_as_trilhas["_trilha_ord"] = df_recall_todas_as_trilhas["trilha"].map(trilha_order_recall).fillna(99)
df_recall_todas_as_trilhas["_familia_ord"] = df_recall_todas_as_trilhas["familia_variacao"].map(familia_order_recall).fillna(99)
df_recall_todas_as_trilhas["_estagio_ord"] = df_recall_todas_as_trilhas["estagio_metrica"].map(estagio_order_recall).fillna(99)
df_recall_todas_as_trilhas["_metrica_ord"] = df_recall_todas_as_trilhas["metrica_recall"].map(metrica_order_recall).fillna(99)

df_recall_todas_as_trilhas = df_recall_todas_as_trilhas.sort_values(
    ["_trilha_ord", "_familia_ord", "componente", "prior", "_estagio_ord", "score_variant", "_metrica_ord"],
    kind="stable",
).drop(columns=["_trilha_ord", "_familia_ord", "_estagio_ord", "_metrica_ord"]).reset_index(drop=True)

display(df_recall_todas_as_trilhas)


## 11. Persistencia e checagens da tabela de Recall@k

Persistimos a tabela consolidada de recall em `outputs_artigo` e registramos checagens minimas de completude, missing e distribuicao por trilha, familia e metrica operacional.


In [ ]:
OUTPUT_RECALL_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)

df_recall_todas_as_trilhas.to_parquet(OUTPUT_RECALL_TABLE_PATH, index=False)
df_recall_todas_as_trilhas.to_csv(OUTPUT_RECALL_CSV_PATH, index=False, encoding="utf-8")

print(f"Tabela de recall salva em: {OUTPUT_RECALL_TABLE_PATH}")
print(f"Tabela de recall em CSV salva em: {OUTPUT_RECALL_CSV_PATH}")
print(f"Shape final da tabela de recall: {df_recall_todas_as_trilhas.shape}")
print("Missing em colunas-chave:")
display(df_recall_todas_as_trilhas[["trilha", "componente", "familia_variacao", "estagio_metrica", "metrica_recall", "valor_recall"]].isna().sum())

print("Contagem por trilha:")
display(df_recall_todas_as_trilhas.groupby("trilha", dropna=False).size().rename("n_linhas").reset_index())

print("Contagem por familia_variacao:")
display(df_recall_todas_as_trilhas.groupby("familia_variacao", dropna=False).size().rename("n_linhas").reset_index())

print("Contagem por metrica_recall:")
display(df_recall_todas_as_trilhas.groupby("metrica_recall", dropna=False).size().rename("n_linhas").reset_index())

print("Resumo final de tipos (df.info):")
df_recall_todas_as_trilhas.info(show_counts=True)


## 12. Analise de sobreposicao dos conjuntos prioritarios

### Objetivo

Comparar o grau de concordancia entre os conjuntos prioritarios gerados pelos tres modelos do artigo a partir dos rankings prontos de `top100k` agricultores unlabeled (`s = 0`).

### Metodologia adotada

- Unidade analitica: `cpf_agricultor`.
- Fonte dos conjuntos: `trilhaA_ranking_unlabeled_top100k.parquet`, `trilhaB_ranking_unlabeled_top100k.parquet` e `trilhaC_ranking_unlabeled_top100k.parquet`.
- Correspondencia com os modelos do artigo:
  - Baseline supervisionado: ranking operacional final da Trilha A (`A4_score_final`).
  - PU classico (Elkan-Noto): ranking oficial bruto da Trilha B (`B2`).
  - nnPU: ranking principal da Trilha C (`C1`).
- Regra de comparacao: a analise trata cada ranking como um conjunto de CPFs priorizados e compara apenas pertencimento ao `top100k`, sem usar a ordem fina dentro do ranking.

### Metricas reportadas

- Percentual de intersecao: proporcao de CPFs comuns entre dois conjuntos. Como todos os rankings tem `100k` casos, esta metrica e diretamente comparavel entre os modelos.
- Indice de Jaccard: intersecao dividida pela uniao dos dois conjuntos. Essa metrica penaliza os casos exclusivos e ajuda a medir concordancia relativa.
- Quantidade de casos comuns e exclusivos: contagem absoluta de CPFs compartilhados e de CPFs presentes apenas em um dos modelos.

### Interpretacao esperada

- Intersecao alta e Jaccard alto indicam forte concordancia sobre quem priorizar.
- Muitos casos exclusivos sugerem que os modelos capturam perfis distintos de agricultores.
- A intersecao tripla identifica o nucleo comum de priorizacao compartilhado pelos tres modelos.


In [ ]:
OUTPUT_OVERLAP_PAIRWISE_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_sobreposicao_top100k_pares.parquet"
OUTPUT_OVERLAP_PAIRWISE_CSV_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_sobreposicao_top100k_pares.csv"
OUTPUT_OVERLAP_BLOCKS_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_sobreposicao_top100k_blocos.parquet"
OUTPUT_OVERLAP_BLOCKS_CSV_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "tabela_sobreposicao_top100k_blocos.csv"
OUTPUT_OVERLAP_BASE_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "base_cpfs_sobreposicao_top100k.parquet"
OUTPUT_OVERLAP_BASE_CSV_PATH = ROOT / "dados" / "dados_processados" / "outputs_artigo" / "base_cpfs_sobreposicao_top100k.csv"

# ---------------------------------------------------------
# Leitura e validacao dos rankings top100k usados no artigo
# ---------------------------------------------------------
#
# Objetivo:
# Confirmar que cada ranking representa 100 mil agricultores unlabeled
# distintos e rastreaveis por CPF. Sem essa checagem, a sobreposicao
# entre modelos pode ser distorcida por duplicidade ou falta de chave.
ranking_specs = [
    {
        "modelo_artigo": "Baseline supervisionado",
        "componente_origem": "A4_score_final",
        "score_usado": "score_final_operacional",
        "arquivo": OUTPUTS_PU_DIR / "trilhaA_ranking_unlabeled_top100k.parquet",
    },
    {
        "modelo_artigo": "PU classico (Elkan-Noto)",
        "componente_origem": "B2",
        "score_usado": "score_pu_raw",
        "arquivo": OUTPUTS_PU_DIR / "trilhaB_ranking_unlabeled_top100k.parquet",
    },
    {
        "modelo_artigo": "nnPU",
        "componente_origem": "C1",
        "score_usado": "score_corrigido",
        "arquivo": OUTPUTS_PU_DIR / "trilhaC_ranking_unlabeled_top100k.parquet",
    },
]

ranking_frames = {}
validation_rows = []

for spec in ranking_specs:
    df_rank = pd.read_parquet(spec["arquivo"]).copy()
    df_rank["cpf_agricultor"] = df_rank["cpf_agricultor"].astype("string")

    if len(df_rank) != 100_000:
        raise ValueError(f"{spec['modelo_artigo']} deveria ter 100000 linhas, mas tem {len(df_rank)}")
    if df_rank["cpf_agricultor"].isna().any():
        raise ValueError(f"{spec['modelo_artigo']} contem cpf_agricultor ausente")
    if df_rank["cpf_agricultor"].nunique(dropna=True) != len(df_rank):
        raise ValueError(f"{spec['modelo_artigo']} contem CPF duplicado no top100k")
    if "s" in df_rank.columns and not df_rank["s"].dropna().isin([0]).all():
        raise ValueError(f"{spec['modelo_artigo']} contem registros fora do conjunto unlabeled (s=0)")

    df_rank = df_rank[["cpf_agricultor"]].copy()
    df_rank["modelo_artigo"] = spec["modelo_artigo"]
    df_rank["componente_origem"] = spec["componente_origem"]
    df_rank["score_usado"] = spec["score_usado"]
    df_rank["rank_posicao"] = pd.RangeIndex(start=1, stop=len(df_rank) + 1, step=1)

    ranking_frames[spec["modelo_artigo"]] = df_rank
    validation_rows.append(
        {
            "modelo_artigo": spec["modelo_artigo"],
            "arquivo": spec["arquivo"].relative_to(ROOT).as_posix(),
            "n_linhas": int(len(df_rank)),
            "n_cpfs_unicos": int(df_rank["cpf_agricultor"].nunique()),
            "coluna_score": spec["score_usado"],
        }
    )

df_overlap_validacao = pd.DataFrame(validation_rows)
print("Validacao dos rankings top100k usados na analise:")
display(df_overlap_validacao)

# ---------------------------------------------------------
# Base-mestra por CPF para auditoria da sobreposicao
# ---------------------------------------------------------
#
# Cada linha representa um CPF presente em pelo menos um dos tres
# rankings. As flags indicam se o agricultor foi priorizado por cada
# modelo, e as colunas de rank guardam a posicao quando o CPF aparece.
base_overlap_cpfs = None
for model_name, df_rank in ranking_frames.items():
    flag_col = f"in_{model_name}"
    rank_col = f"rank_{model_name}"
    df_model = df_rank[["cpf_agricultor", "rank_posicao"]].rename(columns={"rank_posicao": rank_col})
    df_model[flag_col] = True
    base_overlap_cpfs = df_model if base_overlap_cpfs is None else base_overlap_cpfs.merge(df_model, on="cpf_agricultor", how="outer")

for model_name in ranking_frames:
    flag_col = f"in_{model_name}"
    base_overlap_cpfs[flag_col] = base_overlap_cpfs[flag_col].eq(True)

# ---------------------------------------------------------
# Metricas pareadas de concordancia entre os modelos
# ---------------------------------------------------------
#
# A comparacao em pares responde quanto dois modelos concordam sobre
# quem priorizar dentro do top100k. Como os tres conjuntos possuem o
# mesmo tamanho, o percentual de intersecao fica diretamente legivel.
pairwise_rows = []
model_names = list(ranking_frames.keys())
for i, model_1 in enumerate(model_names):
    for model_2 in model_names[i + 1 :]:
        cpfs_1 = set(ranking_frames[model_1]["cpf_agricultor"].astype(str))
        cpfs_2 = set(ranking_frames[model_2]["cpf_agricultor"].astype(str))
        casos_comuns = len(cpfs_1 & cpfs_2)
        casos_exclusivos_1 = len(cpfs_1 - cpfs_2)
        casos_exclusivos_2 = len(cpfs_2 - cpfs_1)
        tamanho_uniao = len(cpfs_1 | cpfs_2)

        pairwise_rows.append(
            {
                "modelo_1": model_1,
                "modelo_2": model_2,
                "n_modelo_1": len(cpfs_1),
                "n_modelo_2": len(cpfs_2),
                "casos_comuns": casos_comuns,
                "percentual_intersecao": casos_comuns / 100_000,
                "indice_jaccard": casos_comuns / tamanho_uniao,
                "casos_exclusivos_modelo_1": casos_exclusivos_1,
                "casos_exclusivos_modelo_2": casos_exclusivos_2,
                "tamanho_uniao": tamanho_uniao,
            }
        )

df_overlap_pairwise = pd.DataFrame(pairwise_rows).sort_values(["modelo_1", "modelo_2"]).reset_index(drop=True)
print("Tabela pareada de sobreposicao dos conjuntos prioritarios:")
display(df_overlap_pairwise)

# ---------------------------------------------------------
# Decomposicao tripla dos conjuntos prioritarios
# ---------------------------------------------------------
#
# Esta tabela separa os CPFs exclusivos e compartilhados entre os tres
# modelos. Ela identifica tanto o nucleo comum de priorizacao quanto as
# parcelas exclusivas de cada abordagem.
block_specs = [
    {
        "bloco_sobreposicao": "somente_baseline",
        "label_legivel": "Somente baseline supervisionado",
        "mask": (
            base_overlap_cpfs["in_Baseline supervisionado"]
            & ~base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & ~base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "somente_pu_classico",
        "label_legivel": "Somente PU classico (Elkan-Noto)",
        "mask": (
            ~base_overlap_cpfs["in_Baseline supervisionado"]
            & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & ~base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "somente_nnpu",
        "label_legivel": "Somente nnPU",
        "mask": (
            ~base_overlap_cpfs["in_Baseline supervisionado"]
            & ~base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "baseline_e_pu_classico",
        "label_legivel": "Baseline e PU classico, sem nnPU",
        "mask": (
            base_overlap_cpfs["in_Baseline supervisionado"]
            & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & ~base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "baseline_e_nnpu",
        "label_legivel": "Baseline e nnPU, sem PU classico",
        "mask": (
            base_overlap_cpfs["in_Baseline supervisionado"]
            & ~base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "pu_classico_e_nnpu",
        "label_legivel": "PU classico e nnPU, sem baseline",
        "mask": (
            ~base_overlap_cpfs["in_Baseline supervisionado"]
            & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & base_overlap_cpfs["in_nnPU"]
        ),
    },
    {
        "bloco_sobreposicao": "baseline_pu_classico_e_nnpu",
        "label_legivel": "Intersecao tripla dos tres modelos",
        "mask": (
            base_overlap_cpfs["in_Baseline supervisionado"]
            & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
            & base_overlap_cpfs["in_nnPU"]
        ),
    },
]

block_rows = []
for block in block_specs:
    quantidade_cpfs = int(block["mask"].sum())
    block_rows.append(
        {
            "bloco_sobreposicao": block["bloco_sobreposicao"],
            "label_legivel": block["label_legivel"],
            "quantidade_cpfs": quantidade_cpfs,
            "percentual_sobre_uniao": quantidade_cpfs / len(base_overlap_cpfs),
            "percentual_sobre_top100k": quantidade_cpfs / 100_000,
        }
    )

df_overlap_blocks = pd.DataFrame(block_rows)
print("Decomposicao tripla da sobreposicao:")
display(df_overlap_blocks)

print("Resumo executivo das sobreposicoes:")
resumo_exec = pd.DataFrame(
    [
        {
            "metrica": "intersecao_tripla_absoluta",
            "valor": int(
                (
                    base_overlap_cpfs["in_Baseline supervisionado"]
                    & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
                    & base_overlap_cpfs["in_nnPU"]
                ).sum()
            ),
        },
        {
            "metrica": "intersecao_tripla_percentual_top100k",
            "valor": float(
                (
                    base_overlap_cpfs["in_Baseline supervisionado"]
                    & base_overlap_cpfs["in_PU classico (Elkan-Noto)"]
                    & base_overlap_cpfs["in_nnPU"]
                ).sum()
                / 100_000
            ),
        },
        {
            "metrica": "tamanho_uniao_tres_modelos",
            "valor": int(len(base_overlap_cpfs)),
        },
    ]
)
display(resumo_exec)

# ---------------------------------------------------------
# Persistencia dos artefatos da analise de sobreposicao
# ---------------------------------------------------------
OUTPUT_OVERLAP_PAIRWISE_PATH.parent.mkdir(parents=True, exist_ok=True)

base_overlap_cpfs.to_parquet(OUTPUT_OVERLAP_BASE_PATH, index=False)
base_overlap_cpfs.to_csv(OUTPUT_OVERLAP_BASE_CSV_PATH, index=False, encoding="utf-8")
df_overlap_pairwise.to_parquet(OUTPUT_OVERLAP_PAIRWISE_PATH, index=False)
df_overlap_pairwise.to_csv(OUTPUT_OVERLAP_PAIRWISE_CSV_PATH, index=False, encoding="utf-8")
df_overlap_blocks.to_parquet(OUTPUT_OVERLAP_BLOCKS_PATH, index=False)
df_overlap_blocks.to_csv(OUTPUT_OVERLAP_BLOCKS_CSV_PATH, index=False, encoding="utf-8")

print(f"Base detalhada por CPF salva em: {OUTPUT_OVERLAP_BASE_PATH}")
print(f"Tabela pareada salva em: {OUTPUT_OVERLAP_PAIRWISE_PATH}")
print(f"Tabela de blocos salva em: {OUTPUT_OVERLAP_BLOCKS_PATH}")
print(f"Shape da base detalhada: {base_overlap_cpfs.shape}")
print("Missing na chave principal:")
display(base_overlap_cpfs[["cpf_agricultor"]].isna().sum())
print("Contagem de CPFs por combinacao de pertencimento:")
display(
    base_overlap_cpfs.groupby(
        [
            "in_Baseline supervisionado",
            "in_PU classico (Elkan-Noto)",
            "in_nnPU",
        ],
        dropna=False,
    ).size().rename("n_cpfs").reset_index()
)
